## 단계 ① ~ ③

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

sales = pd.read_csv("sales_clean.csv")
X = sales[["quantity", "unit_price"]]
y = sales["total_sales"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
# Step 1. 읽기와 나누기 (803, 201)
print("training :", len(X_train), " test :", len(X_test))

training : 803  test : 201


In [2]:
# assign : 기존 데이터프레임을 직접 수정하지 않고, 새로운 column을 추가한 복사본을 만듦
X2_train = X_train.assign(qp=X_train["quantity"] * X_train["unit_price"])
X2_test = X_test.assign(qp=X_test["quantity"] * X_test["unit_price"])
lin = LinearRegression().fit(X_train, y_train)
lin2 = LinearRegression().fit(X2_train, y_train)
tree = DecisionTreeRegressor(random_state=0).fit(X_train, y_train)
# Step 2. model 셋의 test R²
print("linear regression test R² :", round(lin.score(X_test, y_test), 3))
print("linear regression + qp test R² :", round(lin2.score(X2_test, y_test), 3))
print("decision tree (no limit) test R² :", round(tree.score(X_test, y_test), 3))

linear regression test R² : 0.862
linear regression + qp test R² : 1.0
decision tree (no limit) test R² : 0.994


### **단계 ③. 답 (linear regression이 곱셈을 못 하는 이유)**
#### 1. linear regression이 0.862에 그친 이유는?
선형 회귀는 모델이 학습할 수 있는 수식의 형태 자체가 각 변수에 상수를 곱해 더하는 '선형 결합'으로 제한되어 있어 변수까리 곱을 할 수 없다. 즉, 곱셈 항 없이 $w_1x_1 + w_2x_2 + b$라는 평면으로 $z=x_1x_2$라는 곡면을 맞추려고 했기 때문에 구조적 한계로 인해 오차가 발생하게 된다.

#### 2. qp를 더하면 1.0이 되는 이유는?
정답인 total_sales가 애초에 qunatity * unit_price로 계산된 값이므로 선형 회귀 모델이 qp의 가중치를 1로, 나머지 계수로 0으로 학습하면서 예측 오차가 0이 되었기 때문이다.

## 단계 ④ ~ ⑤

In [3]:
rows = []
for dep in [1, 2, 3, 5, 8, 12, None]:
    m = DecisionTreeRegressor(max_depth=dep, random_state=0).fit(X_train, y_train)
    rows.append([str(dep), m.get_n_leaves(),
                 round(m.score(X_train, y_train), 3), round(m.score(X_test, y_test), 3)])
# Step 4. max_depth 별 표
pd.DataFrame(rows, columns=["max_depth", "leaves", "training R²", "test R²"])

,max_depth,leaves,training R²,test R²
0,1,2,0.361,0.346
1,2,4,0.739,0.711
2,3,8,0.876,0.846
3,5,32,0.972,0.961
4,8,250,0.998,0.991
5,12,781,1.000,0.994
6,None,800,1.000,0.994


In [4]:
for dep in [3, 5, 8, 12, None]:
    s = cross_val_score(DecisionTreeRegressor(max_depth=dep, random_state=0), X, y, cv=5)
    # Step 5. cross-validation, max_depth
    print(f"max_depth {str(dep):4s} mean {s.mean():.3f} std {s.std():.3f}")

max_depth 3    mean 0.845 std 0.019
max_depth 5    mean 0.961 std 0.003
max_depth 8    mean 0.992 std 0.000
max_depth 12   mean 0.995 std 0.000
max_depth None mean 0.994 std 0.001


### **단계 ⑤. cross-validation, max_depth 결정과 이유**
#### 1. max_depth 결정
- max_depth = 8

#### 2. 결정 이유
max_depth=12와 비교해 성능($R^2$) 차이가 미미한 수준이므로, 트리 구조가 더 단순해 과적합 위험이 낮고 효율적인 max_depth=8을 선택한다.

#### 추가. traing 1.000과 test 0.994의 차이가 작은 이유
데이터에 무작위 노이즈가 없는 정확한 곱셈 관계여서 트리가 암기할 잡음이 없으며, 2차원 입력 공간에 데이터가 촘촘히 분포해 계단형 분할만으로도 실제 값을 거의 오차 없이 근사하기 때문이다.

## 단계 ⑥ ~ ⑦

In [5]:
import joblib

order = pd.DataFrame([[40, 200000]], columns=["quantity", "unit_price"])
order2 = order.assign(qp=order["quantity"] * order["unit_price"])
tree8 = DecisionTreeRegressor(max_depth=8, random_state=0).fit(X_train, y_train)
print("linear + qp       →", round(lin2.predict(order2)[0]))
print("tree (depth 8)  →", round(tree8.predict(order)[0]))
print("quantity x price  →", 40 * 200000)
joblib.dump(lin2, "sales_model.pkl")          # ⑦ 에서 고른 모델

linear + qp       → 8000000
tree (depth 8)  → 7362301
quantity x price  → 8000000


['sales_model.pkl']

### **단계 ⑦. 어느 model이고, 그렇게 결정한 이유는?**
#### 1. model 결정
- linear + qp (lin2)

#### 2. 결정 이유
실제 정답(quantity x price)인 8,000,000을 기준으로 비교할 때, linear + qp는 실제 값과 완벽히 일치하는 8,000,000(오차 0)을 산출한 반면, tree (depth 8)은 7,362,301을 출력하여 실제 매출보다 637,699(8,000,000 - 7,362,301)만큼 크게 과소 예측하는 오차를 보였기 때문이다.